In [1]:
import cv2
import json
import os
import pandas as pd
from cv_bridge import CvBridge
from bagpy import bagreader
import ast
import numpy as np

In [2]:
# Specify the path to your ROS bag file
bag_path = 'bag_1.bag'

# Create an instance of bagreader
bag = bagreader(bag_path)

# Print the columns of the topic table to understand its structure
print("Columns in topic_table:", bag.topic_table.columns)

# Assuming the correct column for topics is named 'Topics'
topics = bag.topic_table['Topics']  # Adjust 'Topics' if it's named differently

# Now, iterate over each topic and retrieve messages
for topic in topics:
    messages = bag.message_by_topic(topic)
    print(f"Messages from '{topic}' : {messages}")

[INFO]  Successfully created the data folder bag_1.
Columns in topic_table: Index(['Topics', 'Types', 'Message Count', 'Frequency'], dtype='object')
Messages from '/camera/camera_pose' : bag_1/camera-camera_pose.csv
Messages from '/camera/color_image' : bag_1/camera-color_image.csv
Messages from '/camera/joint_states' : bag_1/camera-joint_states.csv


In [3]:
# Initialize CvBridge
bridge = CvBridge()

# Retrieve messages from the specified topic
message = bag.message_by_topic('/camera/color_image')
data = pd.read_csv(message)

# Convert the string representation of bytes directly into a byte array
image_bytes = ast.literal_eval(data.iloc[0]['data'])

# Reshape the byte array to an image array using the dimensions and step provided in the CSV
height, width, step = data.iloc[0]['height'], data.iloc[0]['width'], data.iloc[0]['step']
image_array = np.frombuffer(image_bytes, dtype=np.uint8).reshape((height, width, 3))

# Display the image using OpenCV
cv2.imwrite('camera_extraced_data/Extracted_Image.jpg', image_array)

True

In [4]:
def create_transformation_matrices(df):

    transformation_matrices = []

    row = df.iloc[0]
        
    # Extract quaternion and convert to rotation matrix
    quaternion = [row['orientation.x'], row['orientation.y'], row['orientation.z'], row['orientation.w']]
    
    x, y, z, w = quaternion
    
    rotation_matrix = np.array([
            [1 - 2*y**2 - 2*z**2, 2*x*y - 2*z*w,       2*x*z + 2*y*w],
            [2*x*y + 2*z*w,       1 - 2*x**2 - 2*z**2, 2*y*z - 2*x*w],
            [2*x*z - 2*y*w,       2*y*z + 2*x*w,       1 - 2*x**2 - 2*y**2]
        ])

    # Create the transformation matrix
    transform = np.zeros((4, 4))
    transform[:3, :3] = rotation_matrix
    transform[:3, 3] = row[['position.x', 'position.y', 'position.z']]
    transform[3, 3] = 1

    # Append the matrix to the list
    transformation_matrices.append(transform)
            
    new_df = pd.DataFrame(transformation_matrices[0])
    return new_df

In [5]:

pose=bag.message_by_topic('/camera/camera_pose')
camera_camera_pose_df = pd.read_csv(pose)


new_df = create_transformation_matrices(camera_camera_pose_df)



new_df.to_csv("camera_extraced_data/transformed_camera_pose.csv", index=False, header=False)
